In [16]:
import os
import json
import requests

import pandas as pd
import dotenv
import redis
import numpy as np

In [74]:
# Convert data from byte into datatpyes
def convert_from_byte(byte_dict):
    return {key.decode('utf-8'): value.decode('utf-8') for key, value in byte_dict.items()}

In [17]:
# open .env file and get API keys
env_path = os.path.abspath('../.env.development.local')
dotenv.load_dotenv(env_path)
KV_REST_API_READ_ONLY_TOKEN = os.getenv("KV_REST_API_READ_ONLY_TOKEN")
KV_REST_API_TOKEN = os.getenv("KV_REST_API_TOKEN")
KV_REST_API_URL = os.getenv("KV_REST_API_URL")
KV_URL = os.getenv("KV_URL")

# Set headers for authentication
headers = {
    "Authorization": f"Bearer {KV_REST_API_TOKEN}",
    "Content-Type": "application/json"
}

In [18]:
# Adjust url to work with redis
redis_url = KV_URL
if redis_url.startswith("redis://"):
    redis_url = 'rediss://' + redis_url[len('redis://'):]
r = redis.from_url(redis_url)

In [ ]:
# Import datasets
# datasets = {}
# dataset_names = ['clfever', 'phemeplus', 'vitc']
# for dataset_name in dataset_names:
#     with open(f'{dataset_name}.json') as f:
#         datasets[dataset_name] = json.load(f)

In [28]:
# Import VITC daraset
with open("vitc_evaluation_sup_ref.json") as f:
    vitc = json.load(f)

# Randomise order
np.random.shuffle(vitc)
# Test labels
labels = []
for claim in vitc:
    labels.append(claim['label'])
labels = np.array(labels)
np.unique(labels, return_counts=True)

(array(['REFUTES', 'SUPPORTS'], dtype='<U8'), array([217, 283]))

In [50]:
# Populate Vercel KV with vitc datasets
for datapoint in vitc:
    id = datapoint['claim_id']
    r.hset(id, mapping={
        'claim': datapoint['claim'],
        'evidence': datapoint['evidence'],
        'label': datapoint['label']
    })

In [52]:
vitc_ids = [datapoint['claim_id'] for datapoint in vitc]

In [55]:
# Create three batches for VITC
# 100 samples are included in all batches to test for inter-rater reliability

repeated_samples = vitc_ids[:100]

vitc_batches = {
    'vitc_repeated': repeated_samples 
}
start_index = 100
for i in range(3):
    end_index = start_index + ((len(vitc_ids) - 100) // 3) if i < 2 else len(vitc_ids)
    print(f'{start_index} - {end_index}')
    unique_samples = vitc_ids[start_index:end_index]
    start_index = end_index
    vitc_batches[f'vitc{i+1}_workpackage1'] = unique_samples[:15]
    vitc_batches[f'vitc{i+1}_workpackage3'] = unique_samples[15:]

for key in vitc_batches:
    print(key, len(vitc_batches[key])) 

100 - 233
233 - 366
366 - 500
vitc_repeated 100
vitc1_workpackage1 15
vitc1_workpackage3 118
vitc2_workpackage1 15
vitc2_workpackage3 118
vitc3_workpackage1 15
vitc3_workpackage3 119


In [58]:
# Populate vercel KV with VITC batches
for batch_id in vitc_batches.keys():
    claim_ids = vitc_batches[batch_id]
    r.hset(batch_id, mapping={
        'claim_ids': json.dumps(claim_ids)
    })

In [60]:
# Assign batches to annotators
vitc_annotators = {
    'test': 'vitc1'
}

In [61]:
# Upload annotators to Vercel KV
for annotator_id in vitc_annotators.keys():
    r.hset(annotator_id, mapping={
        'batch_id': vitc_annotators[annotator_id],
        'stage': 'workpackage1',
        'workpackage1_progress': 0,
        'workpackage2_progress': 0,
        'workpackage3_progress': 0
    })

In [80]:
r.hset('test', 'vitc_test', json.dumps(['this is a','test']))

0

In [82]:
test_data = r.hgetall('test')
test_data = convert_from_byte(test_data)
json.loads(test_data['vitc_test'])

['this is a', 'test']

In [15]:
# Get list of all submissions from Prolific (when id wasn't created prior to study)
r.lrange('participants',0,-1)

[]

In [14]:
r.hgetall("66ef9ea77118ba546f2c7c50")

{}

In [274]:
# get ids of participants who completed the task
completed_batches = []
completed_participants = []
submissions = r.lrange('participants',0,-26)
for submission in submissions:
    # convert to dictionary from bytes
    submission = submission.decode('utf-8')
    submission = json.loads(submission)

    if submission["stage"] == "annotation":
        completed_batches.append(submission["batchId"])
        completed_participants.append(submission["participant"])

In [280]:
len(completed_batches)

15

In [275]:
np.unique(completed_batches, return_counts=True)

(array(['batch_vitc_10', 'batch_vitc_11', 'batch_vitc_12', 'batch_vitc_14',
        'batch_vitc_16', 'batch_vitc_17', 'batch_vitc_18', 'batch_vitc_19',
        'batch_vitc_2', 'batch_vitc_3', 'batch_vitc_4', 'batch_vitc_5',
        'batch_vitc_6', 'batch_vitc_8', 'batch_vitc_9'], dtype='<U13'),
 array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))

In [287]:
vitc_queue = [id for id in vitc_batches.keys() if id not in completed_batches]
vitc_queue

['batch_vitc_1',
 'batch_vitc_7',
 'batch_vitc_13',
 'batch_vitc_15',
 'batch_vitc_20']

In [289]:
# delete queue and then only add remaining batches
r.delete('queue')
r.lpush('queue', *vitc_queue)

5

In [9]:
r.rpush('queue', *["batch_vitc_1", "batch_vitc_2", "batch_vitc_3", "batch_vitc_4"])

20

In [11]:
queue = r.lrange('queue', 0, -1)
print(len(queue))
print(queue)

18
[b'batch_vitc_3', b'batch_vitc_4', b'batch_vitc_1', b'batch_vitc_2', b'batch_vitc_3', b'batch_vitc_4', b'batch_vitc_1', b'batch_vitc_2', b'batch_vitc_3', b'batch_vitc_4', b'batch_vitc_1', b'batch_vitc_2', b'batch_vitc_3', b'batch_vitc_4', b'batch_vitc_1', b'batch_vitc_2', b'batch_vitc_3', b'batch_vitc_4']


In [277]:
dataset = []
for participant in completed_participants:
    submission = r.hgetall(participant)
    submission = convert_from_byte(submission)
    for key in submission:
        if "asses" not in key:
            data = vitc_dict[key]
            data['reasoning'] = submission[key]
            data['participant'] = participant
            dataset.append(data)


In [278]:
dataset = pd.DataFrame(dataset)
dataset.to_json('dataset.json', orient='records', lines=True)

In [279]:
dataset['reasoning'].value_counts()

reasoning
deductive    231
abductive    144
Name: count, dtype: int64

## Phemplus

In [ ]:
participant_mapping = {
    'bleiz': 'phemplus1',
    'nelly': 'phemplus2',
    'yazhou': 'phemplus3'
}